In [32]:
import os
import sys
import time
import yaml
import pandas as pd
import json
from string import Template

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

from utils import is_casenum, extract_json

DOCS_DATA_PATH = os.path.join(DATA_PATH, "intermediate_data/cpc", "supplemental_docs.parquet")
ANALYSIS_DATA_PATH = os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data.parquet")

In [33]:
df = pd.read_parquet(ANALYSIS_DATA_PATH)
docs_df = pd.read_parquet(DOCS_DATA_PATH)


In [34]:
docs_df.columns

Index(['date', 'year', 'doc_id', 'text', 'summary', 'referenced_agenda_items',
       'document_type', 'author_type', 'support_or_oppose',
       'support_or_oppose_reasons', 'support_or_oppose_explanation'],
      dtype='str')

In [35]:
df['attached_conditions_text'] = ''
for idx, row in df.iterrows():
    if row['attached_conditions'] != "YES":
        continue
    date = row['date']
    item_no = row['item_no']
    mask = (docs_df['date']==date) & (docs_df['referenced_agenda_items'].apply(lambda x: item_no in x)) & (docs_df['document_type']!="PUBLIC COMMENT") & (docs_df['author_type']=="PUBLIC OFFICIAL") & (docs_df['summary'].str.lower().str.contains("technical"))
    text = ''
    for _, doc in docs_df[mask].iterrows():
        text += doc['text']
    df.at[idx, 'attached_conditions_text'] = text



In [36]:
mask = (df['attached_conditions']=="YES") & (df['attached_conditions_text'].str.len()>0)
print(df.loc[mask].sample(1).iloc[0]['attached_conditions_text'])


                                                                                    
                                                                                    
                                                             Item  Nos. 8-11        
                                                                                    
                                                                                    
                             Department of City Planning                            
                                                                                    
                                                                                    
                                                                                    
                                                                                    
                    City Hall, 200 N. Spring Street, Room 525, Los Angeles, CA 90012
                                                                